In [5]:
import os
import pathlib
import json

import pandas as pd
scf_id = 34
selected_match = 123137
base_path = str(pathlib.Path().resolve().parent)
matches_path = os.path.join(base_path, 'open-data', 'data', 'matches', 'matches_743.json')
with open(matches_path) as f:
    data = json.load(f)
    #save it in dataframe
df_matches = pd.json_normalize(data)
df_matches = df_matches.loc[(df_matches["homeSquadId"] == scf_id) | (df_matches["awaySquadId"] == scf_id)]

players_path = os.path.join(base_path, 'open-data', 'data', 'players', 'players_743.json')
with open(players_path) as f:
    data = json.load(f)
    #save it in dataframe
df_players = pd.json_normalize(data)

# now get the lineup associated with the match
lineup_path = os.path.join(base_path, 'open-data', 'data', 'lineups', f'lineups_{selected_match}.json')
with open(lineup_path) as f:
    data = json.load(f)
    #save it in dataframe
df_lineup = pd.json_normalize(data)

player_kpi_path = os.path.join(base_path, 'open-data', 'data', 'player_kpis', f'player_kpis_{selected_match}.json')
with open(player_kpi_path) as f:
    data = json.load(f)
    #save it in dataframe
df_player_kpi = pd.json_normalize(data)

# merge lineups with player names
name_lookup = df_players.set_index("id")["commonname"].to_dict()
home_players = df_lineup.loc[0, "squadHome.players"]
away_players = df_lineup.loc[0, "squadAway.players"]
df_lineup.at[0, "squadHome.players"] = pd.DataFrame(df_lineup.loc[0, "squadHome.players"]).merge(
    df_players[["id", "commonname"]], on="id", how="left"
)
df_lineup.at[0, "squadAway.players"] = pd.DataFrame(df_lineup.loc[0, "squadAway.players"]).merge(
    df_players[["id", "commonname"]], on="id", how="left"
)

# populate KPI selector
kpi_def_path = os.path.join(base_path, 'open-data', 'data', 'kpi_definitions.json')
with open(kpi_def_path) as f:
    data = json.load(f)
    #save it in dataframe
df_kpi_def = pd.json_normalize(data)
df_kpi_def = df_kpi_def[df_kpi_def["details.label"].notnull()]

In [7]:
playerid = 1294
# df_matches

df_kpi_all = pd.DataFrame()
for id in df_matches.id:
    path = os.path.join(base_path, 'open-data', 'data', 'player_kpis', f'player_kpis_{id}.json')
    with open(path) as f:
        data = json.load(f)
    df_kpi_all = pd.concat([df_kpi_all, pd.json_normalize(data)])

In [12]:
def melt_side(df, players_col, id_col, side):
    d = df[["matchId", id_col, players_col]].rename(columns={id_col: "squadId", players_col: "players"})
    d["side"] = side
    d = d.explode("players").reset_index(drop=True)
    d = pd.concat([d.drop(columns="players"), pd.json_normalize(d["players"])], axis=1)
    d = d.explode("kpis").reset_index(drop=True)
    return pd.concat([d.drop(columns="kpis"), pd.json_normalize(d["kpis"])], axis=1)

long_df = pd.concat([
    melt_side(df_kpi_all, "squadHome.players", "squadHome.id", "home"),
    melt_side(df_kpi_all, "squadAway.players", "squadAway.id", "away"),
], ignore_index=True)

all_match_ids = df_kpi_all["matchId"].tolist()
# Pivot: (playerId, kpiId) als Index, Matches als Spalten
pt = long_df.pivot_table(index=["id", "kpiId"], columns="matchId", values="value", aggfunc="first")
pt = pt.reindex(columns=all_match_ids)  # vollständiges Match-Raster erzwingen

# Pro Zeile die Werte als Liste einsammeln, dann kpiId in Spalten drehen
wide = pt.apply(list, axis=1).unstack("kpiId")
wide.columns = [f"kpi_{int(c)}" for c in wide.columns]

# Statische Spielerinfo dazu
player_info = long_df.drop_duplicates(subset="id", keep="last").set_index("id")[["position", "squadId"]]

players_df = wide.join(player_info).reset_index().rename(columns={"id": "playerId"})
pt
#long_df
#df_kpi_all
#all_match_ids


matchId       122841  122848  122861  122867  122882  122891  122899  122903  \
id     kpiId                                                                   
2      0        28.9     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
       1        15.0     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
       2         7.4     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
       3        26.9     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
       4         5.9     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
...              ...     ...     ...     ...     ...     ...     ...     ...   
118919 1694      NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
       1780      NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
       1781      NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
       1782      NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
       1783      NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   

matchId       122912  122921  ...  123056  123065  123074  123082  123092  \
id     kpiId                  ...                                           
2      0         NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN   
       1         NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN   
       2         NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN   
       3         NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN   
       4         NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN   
...              ...     ...  ...     ...     ...     ...     ...     ...   
118919 1694      NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN   
       1780      NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN   
       1781      NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN   
       1782      NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN   
       1783      NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN   

matchId       123102  123111     123119  123133  123137  
id     kpiId                                             
2      0         NaN     NaN        NaN     NaN     NaN  
       1         NaN     NaN        NaN     NaN     NaN  
       2         NaN     NaN        NaN     NaN     NaN  
       3         NaN     NaN        NaN     NaN     NaN  
       4         NaN     NaN        NaN     NaN     NaN  
...              ...     ...        ...     ...     ...  
118919 1694      NaN     NaN   6.000000     NaN     NaN  
       1780      NaN     NaN   5.000000     NaN     NaN  
       1781      NaN     NaN   1.899901     NaN     NaN  
       1782      NaN     NaN   0.117978     NaN     NaN  
       1783      NaN     NaN  54.233936     NaN     NaN  

[26845 rows x 34 columns]

In [ ]:
players_df[players_df["playerId"] == playerid]["kpi_0"].iloc[0]

[13.7,
 62.2,
 32.5,
 30.8,
 12.4,
 14.3,
 7.9,
 44.8,
 17.4,
 47.1,
 19.5,
 32.4,
 nan,
 18.24,
 25.26,
 nan,
 22.8,
 39.3,
 28.9,
 26.6,
 15.5,
 30.7,
 13.0,
 19.21,
 12.1,
 17.8,
 1.4,
 27.0,
 11.4,
 10.8,
 35.6,
 25.4,
 35.6,
 34.34]